In [ ]:
# Feito por Iorran Valença 
import numpy as np # Para a algebra linear e leitura de matriz
import pandas as pd #receptor de dados
from matplotlib import pyplot as plt #numeros pra identificar

data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
data.head()#dataframe do panda de pixels ate 783

In [ ]:
#Caixa dos dados
data = np.array(data) #especificacao para trabalhar com os arrays do numpy
m, n = data.shape# m = quantidade de linhas/exemplos que se tem (Rows) / n = quantidade de recursos + 1 por causa da coluna de etiqueta (numeraçao e rotulos)
np.random.shuffle(data) #dividindo o leitor de data dos arrays em dev e train para que o lixo de memoria de Desenvolvimento nao entre no treinamento das bias, remove o risco de sobreajuste

#numero de 0 ate N exemplos que serao alocados em data dev.(0 a 1000 pq sim). T e a transposta das matrizes dos pixels pq cada coluna vira um exemplo e nao cada linha (coluna vira exemplo )
data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]
X_dev = X_dev / 255.

 #todo o resto vira dado de treino, transposta  pelo mesmo motivo do dev
data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255.
_,m_train = X_train.shape

In [ ]:
#Recolado tudo e algumas mod das Vn<V4 anteriores
def init_params(): #parametros de iniciaçao
    W1 = np.random.normal(size=(10, 784)) * np.sqrt(1./(784)) #roubei de um blog
    b1 = np.random.normal(size=(10, 1)) * np.sqrt(1./10)
    W2 = np.random.normal(size=(10, 10)) * np.sqrt(1./20)
    b2 = np.random.normal(size=(10, 1)) * np.sqrt(1./(784))
    
    return W1, b1, W2, b2

def ReLU(Z): # Funçao linear X se X>0 e 0 se X<=0, ou seja ela passa por todo elemento de Z, se Z>0 retorna Z se Z<0 retorna 0
    return np.maximum(Z, 0)

def softmax(Z): #Equaçao de ativaçao de funçao softmax
    A = np.exp(Z) / sum(np.exp(Z)) # soma de cada coluna das linhas dividido pelos elementos delas, calculo da probilidade
    return A
    
def forward_prop(W1, b1, W2, b2, X): #propragaçao direta, passagem dos dados da camada de input
    Z1 = W1.dot(X) + b1 #declarando mais variaveis
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z): #Derivada do ReLU que e a equaçao de ativaçao 1
    return Z > 0

def one_hot(Y): #Criador de matriz
    one_hot_Y = np.zeros((Y.size, Y.max() + 1)) #Cria matriz de zeros (parenteses e o tamanho dela)
    one_hot_Y[np.arange(Y.size), Y] = 1 #Cria um array de 0 a M numeros de exemplos de treinamento, diz que pra cada lina va para  a coluna de acesso em Y especificada pelo rotulo, e especifica ela como 1
    one_hot_Y = one_hot_Y.T #Transposta, para transformar cada coluna em exemplo e cada linha em rotulo
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y): # Algoritimo de Backward propagation, conjunto de equaçoes **** Atençao isso da erro lembra, 
    one_hot_Y = one_hot(Y) # equaçoes
    dZ2 = A2 - one_hot_Y #variaveis novas
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1) #a deriv e pra aqui lembra, caso de erro e aqui
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha): #atualizador de parametros
    W1 = W1 - alpha * dW1 #equaçoes dentro das variaveis
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

In [ ]:
#Gradiente descendente ***************** DA MUITO PROBLEMA POR CAUSA DE VARIAVEIS ACIMA NO COMEÇO REVISA TUDO ANTES DE RODAR
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations): #********** DA PAU REVISA AS VARIAVEIS TODAS ABAIXO NA CELULA ACIMA
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0: # NAO BOTAR O I COMO 50 
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 700) # ********************** NAO EXECUTA O TREINAMENTO ANTES DE VERIFICAR AS VARIAVEIS DA FUNÇAO GRADIENT_DESCENT SENAO TRAVA DENOVO 

In [ ]:
#Funçao para fazer os testes de imagem / alçance maximo ate agr : 0.9087804878048781
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    return predictions

def test_prediction(index, W1, b1, W2, b2):
    current_image = X_train[:, index, None]
    prediction = make_predictions(X_train[:, index, None], W1, b1, W2, b2)
    label = Y_train[index]
    print("Prediction: ", prediction)
    print("Label: ", label)
    
    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

In [ ]:
test_prediction(0, W1, b1, W2, b2)
test_prediction(1, W1, b1, W2, b2)
test_prediction(2, W1, b1, W2, b2)
test_prediction(3, W1, b1, W2, b2)